# 2606-Data Ecosystems and Governance in Organizations

# 0. Load Data

In [30]:
from pymongo import MongoClient
from collections import Counter
import pandas as pd
import numpy as np
import json
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

with open('data/raw_credit_applications.json') as f:
    data = json.load(f)

print(f"Total records in raw file: {len(data)}")

Total records in raw file: 502


# 1. Duplicates

In [33]:
# Find duplicates before inserting
ids = [record['_id'] for record in data]
id_counts = Counter(ids)
duplicate_ids = {id_: count for id_, count in id_counts.items() if count > 1}

print(f"Total records in file : {len(data)}")
print(f"Unique _id values     : {len(set(ids))}")
print(f"Duplicate IDs         : {list(duplicate_ids.keys())}")

# Show duplicates
for dup_id in duplicate_ids:
    versions = [r for r in data if r['_id'] == dup_id]
    print(f"\n--- Versions of {dup_id} ---")
    for i, v in enumerate(versions):
        print(f"  Version {i+1}: notes='{v.get('notes', 'none')}', keys={sorted(v.keys())}")

Total records in file : 502
Unique _id values     : 500
Duplicate IDs         : ['app_042', 'app_001']

--- Versions of app_042 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='RESUBMISSION', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']

--- Versions of app_001 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='DUPLICATE_ENTRY_ERROR', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']


**Notes**:

- The duplicate records were first identified when a BulkWriteError occurred during MongoDB ingestion. This demonstrates how database-level constraints can effectively serve as an early data quality gate. The raw dataset contained 502 records, of which 500 were unique applications. app_042 was flagged as a resubmission, and app_001 was flagged as a duplicate entry error. Each was resolved according to its nature: the resubmission retained the most recent version to honor the applicant's latest intent, and the duplicate entry error was resolved by restoring the original record.

In [34]:
# SSNs as unique IDs. (different IDs, same person?)
ssns = [(r['_id'], r['applicant_info'].get('ssn')) for r in data]
ssn_counts = Counter(ssn for _, ssn in ssns if ssn)
dup_ssns = {ssn: count for ssn, count in ssn_counts.items() if count > 1}

print(f"Duplicate SSNs (same SSN across different app IDs): {len(dup_ssns)}")
if dup_ssns:
    for ssn, count in dup_ssns.items():
        app_ids = [id_ for id_, s in ssns if s == ssn]
        print(f"  SSN {ssn} appears in: {app_ids}")

Duplicate SSNs (same SSN across different app IDs): 3
  SSN 652-70-5530 appears in: ['app_042', 'app_042']
  SSN 937-72-8731 appears in: ['app_101', 'app_234']
  SSN 780-24-9300 appears in: ['app_088', 'app_016']


## 1.1 Load into Mongo DB

In [35]:
client = MongoClient()
db = client['novacred']
collection = db['applications']
collection.drop()

for record in data:
    collection.replace_one({'_id': record['_id']}, record, upsert=True)

print(f"Records in MongoDB after upsert: {collection.count_documents({})}")
print("(Duplicate IDs were overwritten with their latest version)")

Records in MongoDB after upsert: 500
(Duplicate IDs were overwritten with their latest version)


## 1.2 Handle Duplicates

In [36]:
# Fix duplicte entry, keep correct version
# app_001 — restore original, discard DUPLICATE_ENTRY_ERROR version
original_app_001 = next(r for r in data 
                        if r['_id'] == 'app_001' 
                        and 'DUPLICATE_ENTRY_ERROR' not in str(r.get('notes', '')))

# Change to correct version
collection.replace_one({'_id': 'app_001'}, original_app_001, upsert=True)

print(f"\nFinal document count in MongoDB: {collection.count_documents({})}")


Final document count in MongoDB: 500
